# Quality audit and limitations

<!-- Drafted with the help of an AI assistant (Claude, Anthropic), disclosed per course rules. -->

How correct is the output? Three checks, from the cheapest to the most expensive: integrity of the tables (done on the whole output), agreement of the tree with the source (done on a sample only) and a reading of random papers (not done by a person yet). The page says which is which.

## 1. Integrity of the tables: whole output

A separate QA notebook did **not** parse again: it verified the SHA-256 of every committed shard against the run's hash manifest, and ran `check_tables` on the 200 shards. The check verifies the column types of the statement, that `element_id` is unique, that every `parent_id` exists, that each version has exactly one root from which every node is reachable, and that a branch has a single parent.

In [ ]:
import json
from pathlib import Path

import pandas as pd

DATA = Path("../data/m2")
check = json.loads((DATA / "check_tables.json").read_text())
status = json.loads((DATA / "qa_status.json").read_text())
papers = pd.read_csv(DATA / "m2_papers.csv", dtype={"arxiv_id": str})
versions = pd.read_csv(DATA / "m2_versions.csv", dtype={"arxiv_id": str})
findings = pd.read_csv(DATA / "audit_findings.csv")
flags = pd.read_csv(DATA / "sentence_flags.csv")

In [ ]:
pd.DataFrame(
    {
        "Value": {
            "Problems found": check["n_problems"],
            "Publications": f"{check['papers']:,}",
            "Versions": f"{check['versions']:,}",
            "Rows of elements": f"{check['rows']['elements']:,}",
            "Rows of hierarchy": f"{check['rows']['hierarchy']:,}",
            "Rows of bibtex": f"{check['rows']['bibtex']:,}",
        }
    }
)

In [ ]:
kinds = pd.Series(check["elements_by_kind"]).sort_values(ascending=False)
kinds.rename("elements").to_frame().assign(share_pct=(100 * kinds / kinds.sum()).round(2))

In [ ]:
roots = int((versions["status"] != "missing").sum())
pd.Series(
    {
        "elements in the tables = elements in m2_papers.csv": check["rows"]["elements"] == int(papers["elements"].sum()),
        "document roots = versions that have a folder": check["elements_by_kind"]["document"] == roots,
        "hierarchy rows = m2_papers.csv": check["rows"]["hierarchy"] == int(papers["hierarchy"].sum()),
        "bibtex rows = m2_papers.csv": check["rows"]["bibtex"] == int(papers["bibtex"].sum()),
    },
    name="agrees",
).to_frame()

`qa_status.json` of the same QA run records what was and was not done:

In [ ]:
pd.Series(status, name='status').to_frame()

## 2 and 3. Agreement with the source and reading of papers: sample only

A subsequent audit `qa-thien-01` compared the official output with source for **2,407 parsed versions from 1,079 papers** (1,114 papers have source available) from the multi-Thiên run; this is a selected subset, not the whole corpus. See [Parsing statistics](02-data-stats.ipynb). An earlier comparison and reading used a separate **sample of 100 publications** (121 versions) parsed with a previous revision of the code. The reading used 30 random publications (seed 2026) and was done with the help of an AI assistant; **the score sheet is still to be filled by a team member**. The table lists what the reading and an automatic scan of the sentences found on that sample, not on the official output.

In [ ]:
findings.set_index('category')

In [ ]:
flags.set_index('scope')

## Known limitations

- **Author macros are not expanded.** A sentence such as `\toolname achieves ...` keeps the macro name; `\newcommand` definitions are dropped, not applied.
- **Template front matter.** The commands of the common journal templates are dropped, but a template the parser has not seen can leave its author block or its notes among the sentences of the first section.
- **References are heuristics.** `citation_key`, `year`, `doi` and `url` are reliable; `authors`, `title`, `journal` and `entry_type` depend on the style of a hand-written `\bibitem`. A share of the parsed versions has no reference at all (see [Parsing statistics](02-data-stats.ipynb)): a `.bbl` that could not be recovered or parsed, or a `.bib` the corpus does not hold.
- **A few parsed versions hold almost nothing** (fewer than 20 elements): a stub root file or an unrecognised structure. They are listed in the statistics and not corrected.
- **Equal elements under one parent are stored once**, so a sentence repeated inside the same section appears once in `hierarchy`, and the order of siblings is the order of the rows, not a column.
- **Not followed:** `\subfile`, `\import` and `\InputIfFileExists`.
- **The structural comparison and the human reading cover a 100-publication sample**, not the 20,000 publications.